# Small RAG Experiment — Semantic Patent Retrieval

This notebook implements a small end-to-end Retrieval-Augmented Generation (RAG)
experiment on the HUPD patent dataset.

The goal is to test whether semantic embeddings can retrieve patents that are
conceptually related to a user's invention idea.

### Pipeline

**Patent Dataset → Text Preparation → Chunking → SPECTER2 Embeddings → FAISS
→ Similarity Search → Top-K Patents → LLM-based Comparison**

## 1. Objective

The objective of this experiment is to build a small prototype of the patent
retrieval pipeline before scaling it to the full dataset.

Each patent will be represented using:

- Title
- Abstract
- Description

Because patent descriptions can be very long, the combined text will be divided
into manageable chunks before generating embeddings.

SPECTER2 will be used as the primary embedding model because it produces
document-level semantic representations suitable for document retrieval.

## 2. Experimental Setup

For the initial experiment, we will use a small subset of the HUPD dataset.

### Components

| Component | Choice |
|---|---|
| Dataset | HUPD |
| Patent fields | Title + Abstract + Description |
| Embedding model | SPECTER2 |
| Embedding dimension | 768 |
| Similarity | Cosine similarity |
| Vector database | FAISS |
| Generation model | Qwen |
| Hardware | CPU-only |

The experiment will initially use a small number of patents so that the complete
pipeline can be tested before scaling up.

In [1]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Imports successful")

Imports successful


## 3. Load the HUPD Dataset

The HUPD dataset has already been downloaded and stored locally.

For this experiment, we will load the dataset from the local project directory
rather than downloading it again from Hugging Face.

Before proceeding with text processing, we will inspect the dataset structure
and verify that the required patent fields are available.

In [2]:
from datasets import load_from_disk

dataset = load_from_disk("../data/sample_dataset")

dataset

Parameter 'format_kwargs'={} of the transform datasets.arrow_dataset.Dataset.set_format couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


DatasetDict({
    train: Dataset({
        features: ['patent_number', 'decision', 'title', 'abstract', 'claims', 'background', 'summary', 'description', 'cpc_label', 'ipc_label', 'filing_date', 'patent_issue_date', 'date_published', 'examiner_id'],
        num_rows: 16153
    })
    validation: Dataset({
        features: ['patent_number', 'decision', 'title', 'abstract', 'claims', 'background', 'summary', 'description', 'cpc_label', 'ipc_label', 'filing_date', 'patent_issue_date', 'date_published', 'examiner_id'],
        num_rows: 9094
    })
})

## 4. Inspect the Dataset

The locally stored HUPD dataset contains separate training and validation
splits.

We will inspect the available splits and their sizes before selecting the
patents for the small retrieval experiment.

In [3]:
print("Dataset splits:")
print(dataset)

print("\nNumber of records:")
for split in dataset:
    print(f"{split}: {len(dataset[split])}")

Dataset splits:
DatasetDict({
    train: Dataset({
        features: ['patent_number', 'decision', 'title', 'abstract', 'claims', 'background', 'summary', 'description', 'cpc_label', 'ipc_label', 'filing_date', 'patent_issue_date', 'date_published', 'examiner_id'],
        num_rows: 16153
    })
    validation: Dataset({
        features: ['patent_number', 'decision', 'title', 'abstract', 'claims', 'background', 'summary', 'description', 'cpc_label', 'ipc_label', 'filing_date', 'patent_issue_date', 'date_published', 'examiner_id'],
        num_rows: 9094
    })
})

Number of records:
train: 16153
validation: 9094


## 5. Inspect the Patent Fields

Before constructing the text representation, we will inspect the available
fields and examine a single patent from the training set.

The retrieval representation for this experiment will use:

**Title + Abstract + Description**

The remaining fields will be retained as metadata for later inspection of
retrieved patents.

In [4]:
train_data = dataset["train"]

print("Available fields:")
print(train_data.column_names)

print("\nFirst patent:")
patent = train_data[0]

print("Patent number:", patent["patent_number"])
print("Title:", patent["title"])
print("Abstract:", patent["abstract"][:500])
print("Description:", patent["description"][:1000])

Available fields:
['patent_number', 'decision', 'title', 'abstract', 'claims', 'background', 'summary', 'description', 'cpc_label', 'ipc_label', 'filing_date', 'patent_issue_date', 'date_published', 'examiner_id']

First patent:
Patent number: 13261748
Title: MINI-OPTICAL NETWORK TERMINAL (ONT)
Abstract: The present invention relates to passive optical network (PON), and in particular, to an optical network terminal (ONT) in the PON system. In one embodiment, the optical network terminal includes a first interface coupled to a communications network, a second interface coupled to a network client and a processor including a memory coupled to the first interface and to the second interface, wherein the processor is capable of converting optical signals to electric signals, such that the network c
Description: FIELD OF THE INVENTION The present invention relates to the field of passive optical network (PON), and in particular, to an optical network terminal (ONT) in the PON system. BACKG

## 6. Analyze Patent Text Length

The retrieval representation will combine the patent's **title, abstract, and
description**.

Patent descriptions can vary considerably in length. Before defining the
chunking strategy, we will measure the character and word counts of these
fields for the training set.

This will help determine an appropriate chunk size for CPU-based embedding
with SPECTER2.

In [5]:
# Calculate text lengths for the three fields

for field in ["title", "abstract", "description"]:
    train_data = train_data.map(
        lambda x: {f"{field}_words": len((x[field] or "").split())}
    )

print("Median word counts:")
for field in ["title", "abstract", "description"]:
    values = train_data[f"{field}_words"]
    print(f"{field:12s}: {np.median(values):,.0f}")

print("\nMaximum word counts:")
for field in ["title", "abstract", "description"]:
    values = train_data[f"{field}_words"]
    print(f"{field:12s}: {np.max(values):,.0f}")

Map:   0%|          | 0/16153 [00:00<?, ? examples/s]

Map:   0%|          | 0/16153 [00:00<?, ? examples/s]

Map:   0%|          | 0/16153 [00:00<?, ? examples/s]

Median word counts:
title       : 7
abstract    : 112
description : 6,977

Maximum word counts:
title       : 56
abstract    : 510
description : 400,946


## 7. Define the Patent Text Representation

Each patent will be represented using its:

**Title + Abstract + Description**

The title and abstract are relatively short, while the description can be
several thousand words long and may be extremely large for some patents.

Therefore, the complete representation cannot be passed to SPECTER2 as a
single input.

We will preserve the title and abstract with the patent text and divide the
long description into smaller chunks before generating embeddings.

In [6]:
# Load the SPECTER2 model and inspect its tokenizer limits

model = SentenceTransformer("allenai/specter2_base")

tokenizer = model.tokenizer

print("Model:", "allenai/specter2_base")
print("Embedding dimension:", model.get_sentence_embedding_dimension())
print("Tokenizer max length:", tokenizer.model_max_length)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model: allenai/specter2_base
Embedding dimension: 768
Tokenizer max length: 512


/tmp/ipykernel_2934/386025142.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


## 8. Define the Chunking Strategy

SPECTER2 accepts a maximum of 512 tokens per input.

To avoid truncating patent information, each patent's combined
**Title + Abstract + Description** will therefore be divided into chunks that
fit within this limit.

The tokenizer will be used to determine chunk boundaries rather than relying
only on character or word counts.

A small overlap between consecutive chunks will be used so that information
near a chunk boundary is not completely separated.

In [7]:
def chunk_text(text, tokenizer, chunk_size=450, overlap=50):
    """
    Split text into token-based chunks.

    chunk_size:
        Number of tokens per chunk. Kept below SPECTER2's 512-token limit.

    overlap:
        Number of tokens shared between consecutive chunks.
    """
    if not text:
        return []

    tokens = tokenizer.encode(
        text,
        add_special_tokens=False
    )

    chunks = []
    start = 0
    step = chunk_size - overlap

    while start < len(tokens):
        chunk_tokens = tokens[start:start + chunk_size]

        chunk = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        chunks.append(chunk)

        if start + chunk_size >= len(tokens):
            break

        start += step

    return chunks


# Build the text representation for the first patent
patent = train_data[0]

patent_text = f"""Title: {patent['title']}

Abstract: {patent['abstract']}

Description: {patent['description']}"""

chunks = chunk_text(
    patent_text,
    tokenizer,
    chunk_size=450,
    overlap=50
)

print("Number of chunks:", len(chunks))
print("First chunk:\n")
print(chunks[0][:1000])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (3988 > 512). Running this sequence through the model will result in indexing errors


Number of chunks: 10
First chunk:

title : mini - optical network terminal ( ont ) abstract : the present invention relates to passive optical network ( pon ), and in particular, to an optical network terminal ( ont ) in the pon system. in one embodiment, the optical network terminal includes a first interface coupled to a communications network, a second interface coupled to a network client and a processor including a memory coupled to the first interface and to the second interface, wherein the processor is capable of converting optical signals to electric signals, such that the network client can access the communications network. description : field of the invention the present invention relates to the field of passive optical network ( pon ), and in particular, to an optical network terminal ( ont ) in the pon system. background of the invention a network interface device permits a subscriber to access a network. a passive optical network ( pon ) is an example of a network capabl

## 9. Verify Chunk Sizes

The chunking function divides the patent text into overlapping token
segments before embedding.

Each chunk should remain below the 512-token input limit of SPECTER2.

We will verify the token length of every generated chunk before passing the
chunks to the embedding model.

In [8]:
# Verify the token length of each chunk

chunk_lengths = [
    len(tokenizer.encode(chunk, add_special_tokens=False))
    for chunk in chunks
]

print("Number of chunks:", len(chunks))
print("Token lengths:", chunk_lengths)
print("Maximum chunk length:", max(chunk_lengths))
print("Minimum chunk length:", min(chunk_lengths))

Number of chunks: 10
Token lengths: [450, 450, 450, 450, 450, 450, 450, 450, 450, 388]
Maximum chunk length: 450
Minimum chunk length: 388


## 10. Refine the Chunking Function

The initial chunking test confirmed that the chunks fit within the SPECTER2
token limit.

We will now refine the chunking function so that long patent text is tokenized
without triggering the tokenizer's maximum-length warning during preprocessing.

The chunk size remains 450 tokens with a 50-token overlap.

In [9]:
def chunk_text(text, tokenizer, chunk_size=450, overlap=50):
    """
    Split text into overlapping token-based chunks without invoking
    the tokenizer's model-length warning on the full document.
    """
    if not text:
        return []

    # Tokenize the full text while explicitly disabling truncation
    tokens = tokenizer(
        text,
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False
    )["input_ids"]

    chunks = []
    step = chunk_size - overlap

    for start in range(0, len(tokens), step):
        chunk_tokens = tokens[start:start + chunk_size]

        if not chunk_tokens:
            break

        chunk = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        chunks.append(chunk)

        if start + chunk_size >= len(tokens):
            break

    return chunks


# Re-create chunks for the first patent
chunks = chunk_text(
    patent_text,
    tokenizer,
    chunk_size=450,
    overlap=50
)

print("Number of chunks:", len(chunks))

chunk_lengths = [
    len(tokenizer.encode(chunk, add_special_tokens=False))
    for chunk in chunks
]

print("Maximum chunk length:", max(chunk_lengths))
print("Minimum chunk length:", min(chunk_lengths))

Number of chunks: 10
Maximum chunk length: 450
Minimum chunk length: 388


## 11. Generate Embeddings for Patent Chunks

Each patent is divided into token-limited chunks before embedding.

SPECTER2 will generate one 768-dimensional embedding for each chunk.

For this initial test, we will generate embeddings for the chunks of a single
patent. This allows us to verify the embedding process before scaling to a
larger number of patents.

In [10]:
# Generate SPECTER2 embeddings for the chunks of the first patent

chunk_embeddings = model.encode(
    chunks,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding type:", type(chunk_embeddings))
print("Embedding shape:", chunk_embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding type: <class 'numpy.ndarray'>
Embedding shape: (10, 768)


## 12. Aggregate Chunk Embeddings into a Patent Embedding

A single patent may produce multiple chunk embeddings because its description can
exceed the input length of SPECTER2.

To perform patent-level retrieval, the chunk embeddings must be combined into a
single vector representing the entire patent.

For this initial experiment, we will use mean pooling across the chunk
embeddings and then normalize the resulting patent embedding.

In [11]:
# Aggregate the chunk embeddings into one patent-level embedding

patent_embedding = np.mean(chunk_embeddings, axis=0)

# Normalize the patent embedding
patent_embedding = patent_embedding / np.linalg.norm(patent_embedding)

print("Patent embedding shape:", patent_embedding.shape)
print("Embedding norm:", np.linalg.norm(patent_embedding))

Patent embedding shape: (768,)
Embedding norm: 1.0


## 13. Select a Small Patent Collection

The single-patent embedding test is successful.

We will now create a small retrieval corpus from the HUPD training set.
Initially, we will use **100 patents** so that the complete retrieval pipeline
can be tested efficiently on CPU.

The validation split will remain separate and will not be included in the
retrieval index.

In [12]:
# Select the first 100 patents from the training set

N_PATENTS = 100

small_corpus = train_data.select(range(N_PATENTS))

print("Number of patents in small corpus:", len(small_corpus))
print("First patent number:", small_corpus[0]["patent_number"])
print("Last patent number:", small_corpus[-1]["patent_number"])

Number of patents in small corpus: 100
First patent number: 13261748
Last patent number: 14888911


## 14. Construct Patent Text

For each patent in the small corpus, we will combine the **Title, Abstract, and
Description** into a single text representation.

These three fields will be preserved with section labels so that the structure
of the patent is retained before the text is divided into chunks.

In [13]:
def build_patent_text(patent):
    """
    Combine the title, abstract, and description into one text representation.
    """
    title = patent["title"] or ""
    abstract = patent["abstract"] or ""
    description = patent["description"] or ""

    return f"""Title: {title}

Abstract: {abstract}

Description: {description}"""


patent_texts = [
    build_patent_text(patent)
    for patent in small_corpus
]

print("Number of patent texts:", len(patent_texts))
print("\nFirst patent text:\n")
print(patent_texts[0][:2000])

Number of patent texts: 100

First patent text:

Title: MINI-OPTICAL NETWORK TERMINAL (ONT)

Abstract: The present invention relates to passive optical network (PON), and in particular, to an optical network terminal (ONT) in the PON system. In one embodiment, the optical network terminal includes a first interface coupled to a communications network, a second interface coupled to a network client and a processor including a memory coupled to the first interface and to the second interface, wherein the processor is capable of converting optical signals to electric signals, such that the network client can access the communications network.

Description: FIELD OF THE INVENTION The present invention relates to the field of passive optical network (PON), and in particular, to an optical network terminal (ONT) in the PON system. BACKGROUND OF THE INVENTION A network interface device permits a subscriber to access a network. A passive optical network (PON) is an example of a network capable

## 15. Chunk the Patent Collection

Each patent text will now be divided into overlapping chunks using the
token-based chunking function defined earlier.

The same chunking parameters will be used for all patents:

- Chunk size: 450 tokens
- Overlap: 50 tokens

This produces manageable inputs for SPECTER2 while preserving some context
between consecutive chunks.

In [14]:
# Chunk all 100 patent texts

all_patent_chunks = []

for patent_text in patent_texts:
    chunks = chunk_text(
        patent_text,
        tokenizer,
        chunk_size=450,
        overlap=50
    )
    all_patent_chunks.append(chunks)

# Basic statistics
num_chunks_per_patent = [len(chunks) for chunks in all_patent_chunks]

print("Number of patents:", len(all_patent_chunks))
print("Total chunks:", sum(num_chunks_per_patent))
print("Minimum chunks per patent:", min(num_chunks_per_patent))
print("Maximum chunks per patent:", max(num_chunks_per_patent))
print("Average chunks per patent:", np.mean(num_chunks_per_patent))

Number of patents: 100
Total chunks: 3207
Minimum chunks per patent: 5
Maximum chunks per patent: 304
Average chunks per patent: 32.07


## 16. Generate Chunk Embeddings

We will now generate SPECTER2 embeddings for all chunks in the small patent
corpus.

Each chunk will produce a 768-dimensional embedding.

Because the experiment is running on a CPU-only machine, the chunks will be
processed in batches to keep memory usage manageable.

The resulting embeddings will later be associated back with their source
patents so that chunk-level results can be aggregated into patent-level
representations.

In [15]:
import time

# Flatten the chunks while keeping track of which patent each chunk belongs to
flat_chunks = []
chunk_patent_ids = []

for patent_idx, chunks in enumerate(all_patent_chunks):
    for chunk in chunks:
        flat_chunks.append(chunk)
        chunk_patent_ids.append(patent_idx)

print("Total chunks:", len(flat_chunks))
print("Chunk-patent mapping:", len(chunk_patent_ids))

# Start timer
start_time = time.perf_counter()

# Generate embeddings in batches
chunk_embeddings_all = model.encode(
    flat_chunks,
    batch_size=8,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Stop timer
elapsed_time = time.perf_counter() - start_time

print("\nEmbedding matrix shape:", chunk_embeddings_all.shape)
print(f"Embedding time: {elapsed_time:.2f} seconds")
print(f"Embedding time: {elapsed_time / 60:.2f} minutes")
print(f"Average time per chunk: {elapsed_time / len(flat_chunks):.4f} seconds")
print(f"Chunks per second: {len(flat_chunks) / elapsed_time:.2f}")

Total chunks: 3207
Chunk-patent mapping: 3207


Batches:   0%|          | 0/401 [00:00<?, ?it/s]


Embedding matrix shape: (3207, 768)
Embedding time: 948.33 seconds
Embedding time: 15.81 minutes
Average time per chunk: 0.2957 seconds
Chunks per second: 3.38


## 17. Verify the Chunk Embeddings

The chunk embeddings have been generated for the 100-patent corpus.

Before proceeding to patent-level aggregation, we will verify:

- the number of embeddings,
- the embedding dimensionality, and
- whether the embeddings are normalized.

This confirms that the generated vectors are suitable for cosine-similarity
based retrieval.

In [16]:
# Verify the generated chunk embeddings

print("Embedding matrix shape:", chunk_embeddings_all.shape)

# Check the norm of the first few embeddings
embedding_norms = np.linalg.norm(chunk_embeddings_all[:5], axis=1)

print("First 5 embedding norms:", embedding_norms)
print("All embeddings finite:", np.isfinite(chunk_embeddings_all).all())

Embedding matrix shape: (3207, 768)
First 5 embedding norms: [1.         1.         0.99999994 1.         1.        ]
All embeddings finite: True


## 18. Save the Chunk Embeddings

The generated chunk embeddings are currently stored only in the Jupyter
kernel's memory.

To avoid recomputing the SPECTER2 embeddings whenever the notebook is
restarted, we will save them to disk.

We will also save the mapping between each chunk and its source patent so
that the embeddings can later be aggregated into patent-level embeddings.

The NumPy format is used for the embedding matrix because it efficiently
stores the numerical vectors.

In [17]:
from pathlib import Path

# Create a directory for saved embeddings
embedding_dir = Path("../data/embeddings")
embedding_dir.mkdir(parents=True, exist_ok=True)

# Save chunk embeddings
np.save(
    embedding_dir / "chunk_embeddings_100.npy",
    chunk_embeddings_all
)

# Save the chunk → patent mapping
np.save(
    embedding_dir / "chunk_patent_ids_100.npy",
    np.array(chunk_patent_ids)
)

print("Saved files:")

print(
    embedding_dir / "chunk_embeddings_100.npy",
    f"({chunk_embeddings_all.shape})"
)

print(
    embedding_dir / "chunk_patent_ids_100.npy",
    f"({len(chunk_patent_ids)},)"
)

Saved files:
../data/embeddings/chunk_embeddings_100.npy ((3207, 768))
../data/embeddings/chunk_patent_ids_100.npy (3207,)


## 19. Aggregate Chunk Embeddings into Patent-Level Embeddings

Each patent may contain multiple chunks, and each chunk has its own
768-dimensional SPECTER2 embedding.

For retrieval, we need one vector per patent.

We will group the chunk embeddings by their source patent, compute the mean
embedding for each patent, and normalize the resulting vector.

This produces a matrix with one 768-dimensional embedding for each patent in
the small corpus.

In [18]:
# Create one embedding for each patent by averaging its chunk embeddings

patent_embeddings = []

for patent_idx in range(len(small_corpus)):
    # Find all chunks belonging to this patent
    mask = np.array(chunk_patent_ids) == patent_idx

    patent_chunk_embeddings = chunk_embeddings_all[mask]

    # Mean pooling across chunks
    patent_embedding = np.mean(patent_chunk_embeddings, axis=0)

    # Normalize
    patent_embedding = patent_embedding / np.linalg.norm(patent_embedding)

    patent_embeddings.append(patent_embedding)

patent_embeddings = np.array(patent_embeddings)

print("Patent embedding matrix shape:", patent_embeddings.shape)

print(
    "First patent embedding norm:",
    np.linalg.norm(patent_embeddings[0])
)

Patent embedding matrix shape: (100, 768)
First patent embedding norm: 1.0


## 20. Save the Patent-Level Embeddings

The chunk embeddings have been aggregated into one normalized 768-dimensional
embedding for each patent.

These patent-level embeddings will be saved to disk and will form the basis
for the vector retrieval index.

In [19]:
# Save patent-level embeddings

np.save(
    embedding_dir / "patent_embeddings_100.npy",
    patent_embeddings
)

print(
    "Saved:",
    embedding_dir / "patent_embeddings_100.npy"
)

print("Shape:", patent_embeddings.shape)

# Verify the saved file can be loaded
saved_embeddings = np.load(
    embedding_dir / "patent_embeddings_100.npy"
)

print("Reloaded shape:", saved_embeddings.shape)

Saved: ../data/embeddings/patent_embeddings_100.npy
Shape: (100, 768)
Reloaded shape: (100, 768)


## 21. Build the FAISS Vector Index

The patent-level embeddings are now available as a matrix of 100 normalized
768-dimensional vectors.

FAISS will be used to create a vector index for efficient similarity search.

Because the embeddings are normalized, inner-product search is equivalent to
cosine similarity for our retrieval experiment.

In [20]:
import faiss

# Ensure embeddings are float32, as expected by FAISS
patent_embeddings_f32 = patent_embeddings.astype("float32")

# Create an inner-product index
index = faiss.IndexFlatIP(patent_embeddings_f32.shape[1])

# Add patent embeddings to the index
index.add(patent_embeddings_f32)

print("FAISS index created successfully")
print("Number of vectors in index:", index.ntotal)
print("Vector dimension:", index.d)

FAISS index created successfully
Number of vectors in index: 100
Vector dimension: 768


## 23. Build the FAISS Vector Index

The patent-level embeddings provide one normalized 768-dimensional vector for
each patent in the retrieval corpus.

We will now add these vectors to a FAISS index using inner-product similarity.
Because the vectors are normalized, inner product is equivalent to cosine
similarity for this retrieval experiment.

The index will contain one vector for each patent.

In [21]:
import faiss

# Use float32 embeddings for FAISS
patent_embeddings_f32 = patent_embeddings.astype("float32")

# Create an inner-product index
index = faiss.IndexFlatIP(patent_embeddings_f32.shape[1])

# Add patent embeddings
index.add(patent_embeddings_f32)

print("FAISS index created successfully")
print("Number of vectors:", index.ntotal)
print("Vector dimension:", index.d)

FAISS index created successfully
Number of vectors: 100
Vector dimension: 768


## 24. Save the FAISS Index

The FAISS index will be saved to disk so that the retrieval system can be
reloaded without rebuilding the index from the patent embeddings.

The index corresponds to the current 100-patent retrieval corpus.

In [22]:
# Save the FAISS index to disk

faiss_index_path = embedding_dir / "patent_index_100.faiss"

faiss.write_index(index, str(faiss_index_path))

print("FAISS index saved to:")
print(faiss_index_path)

# Verify that the saved index can be reloaded
loaded_index = faiss.read_index(str(faiss_index_path))

print("Reloaded vectors:", loaded_index.ntotal)
print("Reloaded dimension:", loaded_index.d)

FAISS index saved to:
../data/embeddings/patent_index_100.faiss
Reloaded vectors: 100
Reloaded dimension: 768


## 25. Define an Invention Query

The retrieval system will accept a new invention idea expressed in natural
language.

The query will be converted into a SPECTER2 embedding using the same embedding
model used for the patent corpus.

This query embedding will then be searched against the FAISS index to identify
the most semantically similar patents.

In [23]:
# Example invention query

query = """
A compact optical network terminal that converts optical signals
to electrical signals and provides network connectivity to multiple
subscriber devices.
"""

print("Invention query:")
print(query)

Invention query:

A compact optical network terminal that converts optical signals
to electrical signals and provides network connectivity to multiple
subscriber devices.



## 26. Generate the Query Embedding

The invention query will be passed through the same SPECTER2 model used to
embed the patent corpus.

The resulting query vector will have 768 dimensions and will be normalized
before being passed to the FAISS index.

In [24]:
# Generate the embedding for the invention query

query_embedding = model.encode(
    query,
    normalize_embeddings=True,
    convert_to_numpy=True
)

print("Query embedding shape:", query_embedding.shape)
print("Query embedding norm:", np.linalg.norm(query_embedding))

Query embedding shape: (768,)
Query embedding norm: 0.99999994


## 27. Retrieve Top-K Similar Patents

The normalized query embedding will be searched against the FAISS index using
inner-product similarity.

The search will return the patents with the highest similarity scores to the
invention query.

For this initial experiment, we will retrieve the **Top-5** patents.

In [25]:
# Search the FAISS index for the most similar patents

TOP_K = 5

# FAISS expects a 2D array: (number_of_queries, embedding_dimension)
query_embedding_f32 = query_embedding.astype("float32").reshape(1, -1)

scores, indices = index.search(
    query_embedding_f32,
    TOP_K
)

print("Similarity scores:")
print(scores[0])

print("\nPatent indices:")
print(indices[0])

Similarity scores:
[0.8280298  0.82756466 0.82126045 0.816982   0.81572986]

Patent indices:
[ 0 17 98 15 51]


## 28. Inspect the Retrieved Patents

The FAISS search returns the positions of the most similar patents within the
small retrieval corpus.

We will map these positions back to the original HUPD records and inspect the
patent number, title, and similarity score for each retrieved result.

The similarity score is used only as a retrieval ranking signal; it should not
be interpreted as a percentage of patent similarity or legal overlap.

In [26]:
# Display the Top-K retrieved patents

retrieved_patents = []

for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
    patent = small_corpus[int(idx)]

    retrieved_patents.append({
        "rank": rank,
        "patent_number": patent["patent_number"],
        "title": patent["title"],
        "similarity_score": float(score)
    })

retrieved_df = pd.DataFrame(retrieved_patents)

retrieved_df

,rank,patent_number,title,similarity_score
0,1,13261748,MINI-OPTICAL NETWORK TERMINAL (ONT),0.828030
1,2,14522409,Burst-Mode Laser Control Circuit and the Metho...,0.827565
2,3,14867910,LUMINESCENT RESONANCE ENERGY TRANSFER SENSORS ...,0.821260
3,4,14437418,Device for Emitting Super-Continuous Wide-Band...,0.816982
4,5,14780334,MULTIPARAMETER DEVICE FOR MEASURING BY OPTICAL...,0.815730


## 29. Inspect Retrieved Patent Content

The Top-K results provide the candidate patents for the invention query.

We will now inspect the retrieved patents in more detail by displaying their
titles and abstracts.

This allows us to perform an initial qualitative check of whether the retrieved
patents are semantically related to the invention query.

In [27]:
# Display the title and abstract of each retrieved patent

for result in retrieved_patents:
    idx = indices[0][result["rank"] - 1]
    patent = small_corpus[int(idx)]

    print("=" * 100)
    print(f"Rank: {result['rank']}")
    print(f"Patent number: {patent['patent_number']}")
    print(f"Similarity score: {result['similarity_score']:.4f}")
    print(f"Title: {patent['title']}")
    print(f"\nAbstract:\n{patent['abstract']}")
    print()

Rank: 1
Patent number: 13261748
Similarity score: 0.8280
Title: MINI-OPTICAL NETWORK TERMINAL (ONT)

Abstract:
The present invention relates to passive optical network (PON), and in particular, to an optical network terminal (ONT) in the PON system. In one embodiment, the optical network terminal includes a first interface coupled to a communications network, a second interface coupled to a network client and a processor including a memory coupled to the first interface and to the second interface, wherein the processor is capable of converting optical signals to electric signals, such that the network client can access the communications network.

Rank: 2
Patent number: 14522409
Similarity score: 0.8276
Title: Burst-Mode Laser Control Circuit and the Method Thereof

Abstract:
A burst-mode laser control circuit and related methods thereof are disclosed. Using an APC loop with an additional burst-mode control circuit, and a switch in series with a diode and in parallel with the laser, a

## 30. Prepare Retrieved Patents for RAG

The retrieved patents will now be prepared as context for the generation stage.

For each retrieved patent, we will keep its patent number, title, abstract, and
description.

At this stage, the goal is only to construct the retrieval context. The language
model will be introduced after the retrieved patent context has been prepared.

In [28]:
# Prepare the retrieved patent context

retrieved_context = []

for result in retrieved_patents:
    idx = indices[0][result["rank"] - 1]
    patent = small_corpus[int(idx)]

    retrieved_context.append({
        "rank": result["rank"],
        "patent_number": patent["patent_number"],
        "title": patent["title"],
        "abstract": patent["abstract"],
        "description": patent["description"],
        "similarity_score": result["similarity_score"]
    })

print("Prepared patents:", len(retrieved_context))

for item in retrieved_context:
    print(
        f"Rank {item['rank']} | "
        f"Patent {item['patent_number']} | "
        f"Score {item['similarity_score']:.4f}"
    )

Prepared patents: 5
Rank 1 | Patent 13261748 | Score 0.8280
Rank 2 | Patent 14522409 | Score 0.8276
Rank 3 | Patent 14867910 | Score 0.8213
Rank 4 | Patent 14437418 | Score 0.8170
Rank 5 | Patent 14780334 | Score 0.8157


## 31. Prepare Compact RAG Context

The Top-5 retrieved patents contain full descriptions that may be much larger
than necessary for the generation stage.

For the initial RAG experiment, we will construct a compact context using the
**title and abstract** of each retrieved patent.

The full patent descriptions remain available in `retrieved_context` and can
be retrieved later when a more detailed comparison is required.

This step keeps the first generation experiment lightweight while preserving
the most concise information about each retrieved patent.

In [29]:
# Build a compact context for the generation model

rag_context_parts = []

for item in retrieved_context:
    part = f"""
Patent {item['rank']}
Patent Number: {item['patent_number']}
Similarity Score: {item['similarity_score']:.4f}
Title: {item['title']}

Abstract:
{item['abstract']}
"""
    rag_context_parts.append(part.strip())

rag_context = "\n\n" + ("\n\n" + "-" * 80 + "\n\n").join(rag_context_parts)

print("RAG context prepared.")
print("Number of retrieved patents:", len(retrieved_context))
print("Context length:", len(rag_context), "characters")

print("\nPreview:\n")
print(rag_context[:4000])

RAG context prepared.
Number of retrieved patents: 5
Context length: 4848 characters

Preview:



Patent 1
Patent Number: 13261748
Similarity Score: 0.8280
Title: MINI-OPTICAL NETWORK TERMINAL (ONT)

Abstract:
The present invention relates to passive optical network (PON), and in particular, to an optical network terminal (ONT) in the PON system. In one embodiment, the optical network terminal includes a first interface coupled to a communications network, a second interface coupled to a network client and a processor including a memory coupled to the first interface and to the second interface, wherein the processor is capable of converting optical signals to electric signals, such that the network client can access the communications network.

--------------------------------------------------------------------------------

Patent 2
Patent Number: 14522409
Similarity Score: 0.8276
Title: Burst-Mode Laser Control Circuit and the Method Thereof

Abstract:
A burst-mode laser control cir

## 32. Connect to the Local Generation Model

The retrieval stage has identified the most semantically similar patents and
prepared a compact context from their titles and abstracts.

The next stage is to pass the invention query and retrieved patent context to
a locally hosted language model.

The model will be accessed through the local Ollama API so that patent text
remains on the local system.

For the initial experiment, the generation model will be Qwen.

In [32]:
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"
GENERATION_MODEL = "qwen3.5:4b"

# Check that the local Ollama API is reachable
response = requests.get("http://localhost:11434/api/tags")

print("Ollama status:", response.status_code)

if response.ok:
    models = response.json().get("models", [])
    print("Available models:")
    for model_info in models:
        print("-", model_info["name"])
else:
    print(response.text)

Ollama status: 200
Available models:
- gemma3:4b
- deepseek-r1:8b
- qwen3.5:4b
- nomic-embed-text:latest
- qwen2.5-coder:3b


## 33. Generate a RAG Response

The invention query and the retrieved patent context will be sent to the
local Qwen model through the Ollama API.

The model will compare the proposed invention with the retrieved patents and
identify the main similarities and differences.

The response will be based only on the retrieved patent context supplied in
the prompt.

In [34]:
# Generate the RAG response using streaming

rag_prompt = f"""
You are assisting with semantic patent prior-art analysis.

User's invention idea:
{query}

Retrieved patents:
{rag_context}

For each patent, briefly state:
1. Main technical similarity
2. Main technical difference
3. Relevance: High, Medium, or Low

Finally, identify the single most relevant patent.

Do not make legal conclusions about infringement or patent validity.
Use only the information provided in the retrieved patent context.
"""

response = requests.post(
    OLLAMA_URL,
    json={
        "model": GENERATION_MODEL,
        "prompt": rag_prompt,
        "stream": True,
        "options": {
            "num_predict": 250
        }
    },
    stream=True,
    timeout=60
)

response.raise_for_status()

rag_answer_parts = []

for line in response.iter_lines():
    if line:
        data = line.decode("utf-8")
        result = __import__("json").loads(data)

        if "response" in result:
            text = result["response"]
            rag_answer_parts.append(text)
            print(text, end="", flush=True)

        if result.get("done", False):
            break

rag_answer = "".join(rag_answer_parts)

print("\n\nRAG generation complete.")



RAG generation complete.


## 34. Save the RAG Experiment Result

The current experiment contains several outputs that should be preserved:

- The invention query
- The retrieved patent numbers
- Their similarity scores
- The retrieved patent titles
- The generated RAG response

These results will be saved as a JSON file so that the experiment can be
reviewed later without rerunning the retrieval and generation stages.

In [35]:
import json

# Collect the experiment results
experiment_result = {
    "model": {
        "embedding": "allenai/specter2_base",
        "generation": GENERATION_MODEL
    },
    "query": query,
    "top_k": TOP_K,
    "retrieved_patents": retrieved_patents,
    "rag_answer": rag_answer
}

# Save the result
result_path = embedding_dir / "smallrag_experiment_100.json"

with open(result_path, "w", encoding="utf-8") as f:
    json.dump(experiment_result, f, indent=2, ensure_ascii=False)

print("Experiment result saved to:")
print(result_path)

Experiment result saved to:
../data/embeddings/smallrag_experiment_100.json


## 35. Evaluate the Retrieved Patents

The first RAG run has produced a ranked list of retrieved patents.

For this initial experiment, we will perform a qualitative evaluation of the
Top-5 results.

The retrieved patents will be examined for:

- Technical relevance to the invention query
- Meaningful semantic similarity
- Whether the ranking appears reasonable
- Obvious irrelevant results

This provides an initial sanity check of the retrieval system before increasing
the corpus size.

In [36]:
# Display the Top-5 results in a compact evaluation table

evaluation_df = retrieved_df[
    ["rank", "patent_number", "title", "similarity_score"]
].copy()

evaluation_df["similarity_score"] = evaluation_df["similarity_score"].round(4)

evaluation_df

,rank,patent_number,title,similarity_score
0,1,13261748,MINI-OPTICAL NETWORK TERMINAL (ONT),0.8280
1,2,14522409,Burst-Mode Laser Control Circuit and the Metho...,0.8276
2,3,14867910,LUMINESCENT RESONANCE ENERGY TRANSFER SENSORS ...,0.8213
3,4,14437418,Device for Emitting Super-Continuous Wide-Band...,0.8170
4,5,14780334,MULTIPARAMETER DEVICE FOR MEASURING BY OPTICAL...,0.8157


## 36. Initial Retrieval Evaluation

A single query is not sufficient to judge the retrieval behavior of the
prototype.

For an initial qualitative evaluation, we will test the system with multiple
invention queries covering different technical areas.

For each query, the Top-5 retrieved patents will be manually evaluated as:

- **Relevant** — technically related to the proposed invention.
- **Partially relevant** — shares some concepts or components but is not closely
  related.
- **Irrelevant** — does not have meaningful technical relevance.

This evaluation is intended as an initial sanity check rather than a formal
benchmark, since no ground-truth relevance labels have yet been established.

In [37]:
# Define a small set of evaluation queries from different technical areas

evaluation_queries = [
    """
    A compact optical network terminal that converts optical signals
    to electrical signals and provides network connectivity to multiple
    subscriber devices.
    """,

    """
    A system that uses machine learning to detect diseases in plants
    from photographs captured by a camera.
    """,

    """
    A wearable device that continuously monitors a person's body
    temperature and transmits measurements wirelessly to a mobile device.
    """,

    """
    An autonomous robotic vehicle that uses cameras and sensors to
    navigate indoors and avoid obstacles.
    """,

    """
    A battery management system that monitors individual battery cells
    and balances their charge to improve battery life and safety.
    """
]

print("Number of evaluation queries:", len(evaluation_queries))

for i, q in enumerate(evaluation_queries, start=1):
    print(f"\nQuery {i}:")
    print(q.strip())

Number of evaluation queries: 5

Query 1:
A compact optical network terminal that converts optical signals
    to electrical signals and provides network connectivity to multiple
    subscriber devices.

Query 2:
A system that uses machine learning to detect diseases in plants
    from photographs captured by a camera.

Query 3:
A wearable device that continuously monitors a person's body
    temperature and transmits measurements wirelessly to a mobile device.

Query 4:
An autonomous robotic vehicle that uses cameras and sensors to
    navigate indoors and avoid obstacles.

Query 5:
A battery management system that monitors individual battery cells
    and balances their charge to improve battery life and safety.


## 37. Retrieve Top-K Results for Evaluation Queries

Each evaluation query will be embedded using SPECTER2 and searched against the
same FAISS index containing the 100-patent retrieval corpus.

For consistency, the same Top-5 retrieval setting will be used for every query.

The resulting table will contain the query number, rank, patent number, title,
and similarity score.

In [38]:
# Retrieve Top-5 patents for each evaluation query

evaluation_results = []

for query_id, evaluation_query in enumerate(evaluation_queries, start=1):
    # Generate query embedding
    query_embedding_eval = model.encode(
        evaluation_query,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    # FAISS expects a 2D float32 array
    query_embedding_eval = (
        query_embedding_eval
        .astype("float32")
        .reshape(1, -1)
    )

    # Search the index
    scores_eval, indices_eval = index.search(
        query_embedding_eval,
        TOP_K
    )

    # Store results
    for rank, (idx, score) in enumerate(
        zip(indices_eval[0], scores_eval[0]),
        start=1
    ):
        patent = small_corpus[int(idx)]

        evaluation_results.append({
            "query_id": query_id,
            "rank": rank,
            "patent_number": patent["patent_number"],
            "title": patent["title"],
            "similarity_score": float(score)
        })

evaluation_results_df = pd.DataFrame(evaluation_results)

evaluation_results_df["similarity_score"] = (
    evaluation_results_df["similarity_score"].round(4)
)

evaluation_results_df

,query_id,rank,patent_number,title,similarity_score
0,1,1,13261748,MINI-OPTICAL NETWORK TERMINAL (ONT),0.8280
1,1,2,14522409,Burst-Mode Laser Control Circuit and the Metho...,0.8276
2,1,3,14867910,LUMINESCENT RESONANCE ENERGY TRANSFER SENSORS ...,0.8213
3,1,4,14437418,Device for Emitting Super-Continuous Wide-Band...,0.8170
4,1,5,14780334,MULTIPARAMETER DEVICE FOR MEASURING BY OPTICAL...,0.8157
5,2,1,14867910,LUMINESCENT RESONANCE ENERGY TRANSFER SENSORS ...,0.8167
6,2,2,14780334,MULTIPARAMETER DEVICE FOR MEASURING BY OPTICAL...,0.8008
7,2,3,14782449,SYSTEMS AND METHODS TO ASSES MICROBIOMES AND T...,0.8001
8,2,4,14779497,PRIVACY MASKING METHOD,0.7902
9,2,5,14241799,PORTABLE DRUG DISPENSER,0.7806
